# Week 1 Day 2 Lab - UnderCover: the Agent (free version)

A claims handler at UnderCover Travel Insurance has already decided each claim in its queue. Here a LangChain agent, running on `openrouter/free`, decides the same claims using tools: it looks up the policy, calculates the baggage payout and records its decision. Then we compare its answers with the handlers' decisions.

In [ ]:
import json
import os
import time
from pathlib import Path
from typing import Literal

import pandas as pd
import plotly.express as px
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain.agents import create_agent
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from openai import OpenAI
from pydantic import BaseModel, Field

## Connect to OpenRouter - and please read note here

The OpenRouter key comes from the `.env` file you made on day 1 in the `undercover` folder.

### NOTE:

If this is a new OpenRouter account, then you might get an error at some point from OpenRouter like: "new accounts are limited to 20 requests per minute for this model"

If you get that error, then add this line immediately after calling an LLM:

`time.sleep(3)`

and you should be in good shape.

In [ ]:
load_dotenv(override=True)
openrouter_api_key = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"

if openrouter_api_key:
    print("OpenRouter API key found")
else:
    print("No OpenRouter API key found. Add OPENROUTER_API_KEY to the .env file")

openrouter = OpenAI(base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

## Load the data

`TINY = True` keeps 5 of the 25 claims, which fits the free daily limit.

In [ ]:
DATA = Path("../../../data")


def load(name):
    """Read one of the JSON seed files."""
    return json.loads((DATA / name).read_text(encoding="utf-8"))


def get_sample_data(tiny=False, held_out=False):
    """Return a list of claims, and the handlers' decisions on them keyed by claim id.

    held_out=True gives the twenty-five held-out claims in place of the queue.
    tiny=True keeps the five claims marked for the tiny version of either set.
    """
    claims = load("evals/held_out_claims.json" if held_out else "claims.json")
    decisions = load("evals/held_out_decisions.json" if held_out else "evals/decisions.json")
    if tiny:
        chosen = {decision["claim_id"] for decision in decisions if decision["tiny"]}
        claims = [claim for claim in claims if claim["claim_id"] in chosen]
        decisions = [decision for decision in decisions if decision["tiny"]]
    labels = {decision["claim_id"]: decision for decision in decisions}
    return claims, labels

In [ ]:
TINY = True

claims, labels = get_sample_data(tiny=TINY)
policies = {policy["policy_id"]: policy for policy in load("policies.json")}
policy_wording = (DATA / "policy_wording.md").read_text(encoding="utf-8")

print(f"{len(claims)} claims, {len(labels)} labels, {len(policies)} policies")
print(f"The policy wording is {len(policy_wording.split()):,} words")

## Instructions and scoring helpers

`INSTRUCTIONS` goes into the agent's system prompt. `is_right` compares an answer with the handler's decision and `show_misses` shows the claims the agent got wrong.

In [ ]:
INSTRUCTIONS = """You are a claims handler at UnderCover Travel Insurance.
Decide the claim you are given, using the policy wording below and the customer's policy record.

Choose one decision:
- approve: the wording covers the claim and the required documents are attached. Work out the payout using section 10.
- decline: the wording does not cover the claim.
- refer: a colleague needs to look at it, because a required document is missing, or the story and the documents disagree, or the wording does not settle the question.
"""

In [ ]:
def is_right(answer, label):
    """True when the decision matches the handler's and the payout is within a dollar."""
    return answer["decision"] == label["decision"] and abs(answer["payout"] - label["payout"]) <= 1


def show_misses(results, labels):
    """Show each claim in a set of results that the model got wrong, with its rationale and the handler's reason."""
    for result in results:
        if not result["right"]:
            label = labels[result["claim_id"]]
            display(Markdown(f"""**{result['claim_id']}, {label['difficulty']}**: {result['decision']} ${result['payout']:,.2f}, where the handler said {label['decision']} ${label['payout']:,.2f}

- Model: {result['rationale']}
- Handler: {label['reason']}
- Tests: {label['tests']}
"""))


## Part 3: Using LangChain create_agent

Our approaches so far have used a single LLM call with Structured Outputs.

Now we'll use a different approach: an AI agent.

An LLM, with tools, in a loop, to achieve a goal!

We'll use LangChain's `create_agent` to run the loop for us.

We give it 2 tools. `get_policy` looks up the customer's policy record. `record_decision` saves the decision.

In [ ]:
recorded = {}


DEVICE_TYPES = Literal["phone", "tablet", "laptop", "camera", "lens", "e-reader", "smartwatch", "other"]


class BaggageItem(BaseModel):
    name: str = Field(description="Short name of the item")
    claimed: float = Field(description="Amount the customer claims for this item, in USD")
    device_type: DEVICE_TYPES = Field(
        description="The item's own type. Use 'other' for anything not in this list, "
                    "even if the claim form calls it electronics (headphones, game console, etc.)."
    )


@tool
def calculate_baggage_payout(
    items: list[BaggageItem],
    plan_limit: float,
    addon_limit: float,
    has_electronics_plus: bool,
    deductible: float,
    paid_by_others: float = 0.0,
) -> str:
    """Compute the baggage payout. plan_limit is the plan's per-item limit (from the wording);
    addon_limit is the Electronics Plus per-item limit (0 if the policy lacks the add-on)."""
    eligible, lines = 0.0, []
    for i in items:
        limit = addon_limit if (has_electronics_plus and i.device_type != "other") else plan_limit
        counted = min(i.claimed, limit)
        eligible += counted
        lines.append(f"{i.name}: claimed {i.claimed}, limit {limit}, counted {counted}")
    payout = max(eligible - paid_by_others - deductible, 0)
    return json.dumps({"lines": lines, "eligible": eligible, "payout": payout})


@tool
def get_policy(policy_id: str) -> str:
    """Look up a customer's policy record: the plan, the add-ons, the purchase date, the trip and the travelers."""
    policy = policies.get(policy_id)
    return json.dumps(policy, indent=2) if policy else f"There is no policy {policy_id}"


@tool
def record_decision(claim_id: str, decision: Literal["approve", "decline", "refer"], payout: float, rationale: str) -> str:
    """Record the decision on a claim in the claims system.

    payout is the amount to pay in US dollars, or 0 for a decline or a referral.
    rationale is 2-3 sentences to explain the decision along with the relevant section of the policy.
    """
    recorded[claim_id] = {"decision": decision, "payout": payout, "rationale": rationale}
    return f"Recorded {decision} on {claim_id}"

The agent uses `openrouter/free` through LangChain's `ChatOpenAI`, pointed at OpenRouter. Its system prompt has the same instructions and policy wording as before, plus a line about the tools. Once the agent has looked up the policy it has the same information as before, so we can compare its score with the single calls in Part 2.

In [ ]:
llm = ChatOpenAI(model="openrouter/free", base_url=OPENROUTER_BASE_URL, api_key=openrouter_api_key, timeout=120, max_retries=5)

claims_manual = (DATA / "claims_manual.md").read_text(encoding="utf-8")



AGENT_SYSTEM_PROMPT = f"""{INSTRUCTIONS}
Use your tools to look up the customer's policy record and to record your decision.

For baggage claims: list every item, classify each by its own type (not the claim form's
category), call calculate_baggage_payout, use its payout, and include its line-by-line result
in your rationale. An add-on changes a limit only for what its definition in the wording lists.


{policy_wording}

{claims_manual}
"""


def make_claim_prompt(claim):
    """The claim alone, as JSON. The agent looks up the policy record for itself."""
    return f"""# Claim

{json.dumps(claim, indent=2)}
"""


agent = create_agent(model=llm, tools=[get_policy, record_decision,calculate_baggage_payout], system_prompt=AGENT_SYSTEM_PROMPT)

First, trying out one claim. The model asks for the policy, the tool returns it, the model records its decision and then finishes with a short message.

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claims[0])}]})

for message in result["messages"][1:]:
    message.pretty_print()

Now all the claims. Each claim takes three model calls: one to ask for the policy, one to record the decision and one to finish. A claim with no recorded decision counts as wrong.

In [ ]:
def decide_with_agent(claim):
    """Run the agent on one claim and check what it recorded against the handler's decision."""
    start = time.time()
    result = agent.invoke({"messages": [{"role": "user", "content": make_claim_prompt(claim)}]})
    answer = recorded.get(claim["claim_id"], {"decision": "nothing recorded", "payout": 0.0, "rationale": ""})
    label = labels[claim["claim_id"]]
    return {
        "model": "openrouter/free",
        "technique": "Agent",
        "claim_id": claim["claim_id"],
        **answer,
        "difficulty": label["difficulty"],
        "right": is_right(answer, label),
        "seconds": round(time.time() - start, 1),
    }


recorded.clear()
agent_results = []
for claim in claims:
    agent_results.append(decide_with_agent(claim))

print(f"{len(recorded)} of {len(claims)} claims have a recorded decision")

In [ ]:
pd.DataFrame(agent_results).groupby(["model", "technique"], sort=False)[["right", "seconds"]].sum()

In [ ]:
show_misses(agent_results, labels)

## Summary

Accuracy and average time for one claim.

In [ ]:
rows = []
for results in [agent_results]:
    right = sum(result["right"] for result in results)
    seconds = sum(result["seconds"] for result in results)
    rows.append({
        "LLM": results[0]["model"],
        "technique": results[0]["technique"],
        "accuracy": f"{right} / {len(results)}",
        "speed": f"{seconds / len(results):.1f} seconds a claim",
    })

pd.DataFrame(rows)